In [1]:
import pandas as pd

df = pd.read_csv(r'C:\BFSI_Loan_Analytics\data\processed\lending_club_cleaned.csv', low_memory=False)

print("Shape:", df.shape)
print("Nulls:", df.isnull().sum().sum())

Shape: (1342942, 86)
Nulls: 0


In [2]:
# Create binary target column
df['loan_outcome'] = df['loan_status'].map({'Charged Off': 1, 'Fully Paid': 0})

print("loan_outcome value counts:")
print(df['loan_outcome'].value_counts())
print("\nNull check:", df['loan_outcome'].isnull().sum())
print("Default rate:", round(df['loan_outcome'].mean() * 100, 2), "%")

loan_outcome value counts:
loan_outcome
0    1076706
1     266236
Name: count, dtype: int64

Null check: 0
Default rate: 19.82 %


In [3]:
# Create DTI bands
df['DTIBand'] = pd.cut(
    df['dti'],
    bins=[0, 10, 15, 20, 25, 100],
    labels=['0-10', '10-15', '15-20', '20-25', '25+']
)

print("DTIBand value counts:")
print(df['DTIBand'].value_counts().sort_index())
print("\nNull check:", df['DTIBand'].isnull().sum())

DTIBand value counts:
DTIBand
0-10     244678
10-15    272719
15-20    288922
20-25    240633
25+      294621
Name: count, dtype: int64

Null check: 1369


In [4]:
# Check what dti values are causing nulls in DTIBand
print("DTI values that fell outside bins:")
print(df[df['DTIBand'].isnull()]['dti'].describe())
print("\nNegative DTI count:", (df['dti'] < 0).sum())
print("DTI = 0 count:", (df['dti'] == 0).sum())
print("DTI > 100 count:", (df['dti'] > 100).sum())

DTI values that fell outside bins:
count    1369.000000
mean      103.212900
std       203.209217
min        -1.000000
25%         0.000000
50%         0.000000
75%       132.290000
max       999.000000
Name: dti, dtype: float64

Negative DTI count: 2
DTI = 0 count: 836
DTI > 100 count: 531


In [5]:
# Fix DTIBand — handle 0, negative, and >100 values
df['dti_capped'] = df['dti'].clip(lower=0, upper=100)

df['DTIBand'] = pd.cut(
    df['dti_capped'],
    bins=[-0.01, 10, 15, 20, 25, 100],
    labels=['0-10', '10-15', '15-20', '20-25', '25+']
)

print("DTIBand value counts:")
print(df['DTIBand'].value_counts().sort_index())
print("\nNull check:", df['DTIBand'].isnull().sum())

# Drop the temporary column
df.drop(columns=['dti_capped'], inplace=True)

DTIBand value counts:
DTIBand
0-10     245516
10-15    272719
15-20    288922
20-25    240633
25+      295152
Name: count, dtype: int64

Null check: 0


In [6]:
# Create income bands using quartiles
df['IncomeBand'] = pd.qcut(
    df['annual_inc'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

print("IncomeBand value counts:")
print(df['IncomeBand'].value_counts().sort_index())

print("\nIncome quartile ranges:")
print(pd.qcut(df['annual_inc'], q=4).value_counts().sort_index())

print("\nNull check:", df['IncomeBand'].isnull().sum())

IncomeBand value counts:
IncomeBand
Low          335744
Medium       361590
High         311994
Very High    333614
Name: count, dtype: int64

Income quartile ranges:
annual_inc
(-0.001, 45800.0]        335744
(45800.0, 65000.0]       361590
(65000.0, 90000.0]       311994
(90000.0, 10999200.0]    333614
Name: count, dtype: int64

Null check: 0


In [7]:
# Create loan size bands
df['LoanSizeBand'] = pd.cut(
    df['loan_amnt'],
    bins=[0, 10000, 25000, 40000],
    labels=['Small', 'Medium', 'Large']
)

print("LoanSizeBand value counts:")
print(df['LoanSizeBand'].value_counts().sort_index())
print("\nNull check:", df['LoanSizeBand'].isnull().sum())

LoanSizeBand value counts:
LoanSizeBand
Small     554506
Medium    630045
Large     158391
Name: count, dtype: int64

Null check: 0


In [8]:
# Create risk tier based on loan grade
def assign_risk_tier(grade):
    if grade in ['A', 'B']:
        return 'Low'
    elif grade in ['C', 'D']:
        return 'Medium'
    elif grade in ['E', 'F', 'G']:
        return 'High'
    else:
        return 'Unknown'

df['RiskTier'] = df['grade'].apply(assign_risk_tier)

print("RiskTier value counts:")
print(df['RiskTier'].value_counts())
print("\nGrade distribution for verification:")
print(df.groupby(['RiskTier', 'grade']).size().reset_index(name='count').to_string())
print("\nNull check:", df['RiskTier'].isnull().sum())

RiskTier value counts:
RiskTier
Low       627234
Medium    581467
High      134241
Name: count, dtype: int64

Grade distribution for verification:
  RiskTier grade   count
0     High     E   93275
1     High     F   31912
2     High     G    9054
3      Low     A  234912
4      Low     B  392322
5   Medium     C  381070
6   Medium     D  200397

Null check: 0


In [9]:
# Final check — all 5 engineered columns
engineered_cols = ['loan_outcome', 'DTIBand', 'IncomeBand', 'LoanSizeBand', 'RiskTier']

print("Shape:", df.shape)
print("\nEngineered columns null check:")
for col in engineered_cols:
    print(f"  {col}: {df[col].isnull().sum()} nulls")

print("\nSample of engineered columns:")
print(df[engineered_cols].head(10).to_string())

Shape: (1342942, 91)

Engineered columns null check:
  loan_outcome: 0 nulls
  DTIBand: 0 nulls
  IncomeBand: 0 nulls
  LoanSizeBand: 0 nulls
  RiskTier: 0 nulls

Sample of engineered columns:
   loan_outcome DTIBand IncomeBand LoanSizeBand RiskTier
0             0    0-10     Medium        Small   Medium
1             0   15-20     Medium       Medium   Medium
2             0   10-15     Medium       Medium      Low
3             0     25+  Very High       Medium     High
4             0   10-15        Low       Medium   Medium
5             0   10-15  Very High       Medium      Low
6             0   15-20       High       Medium      Low
7             0   10-15       High        Small      Low
8             0     25+        Low        Small      Low
9             0     25+     Medium        Small   Medium


In [10]:
output_path = r'C:\BFSI_Loan_Analytics\data\processed\lending_club_features.csv'
df.to_csv(output_path, index=False)

print(f"File saved to: {output_path}")
print(f"Final shape: {df.shape}")

File saved to: C:\BFSI_Loan_Analytics\data\processed\lending_club_features.csv
Final shape: (1342942, 91)
